In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType

def gerar_treino(spark, n=10000):
    np.random.seed(42)
    
    # Função auxiliar para One-Hot Encoding das regiões
    regioes = np.random.choice(['nordeste', 'norte', 'sudeste', 'sul', 'outro'], n, p=[0.2, 0.1, 0.4, 0.2, 0.1])
    
    pdf_treino = pd.DataFrame({
        "renda_mensal_k": np.random.uniform(700.0, 25000, n),
        "tempo_medio_clique_segundos": np.random.uniform(0, 150, n),
        "media_interacoes_suporte": np.random.uniform(0, 150, n),
        "media_cupons_ativos": np.random.uniform(0, 15, n),
        "media_score_nps_cliente": np.random.randint(0, 10, n),
        "media_dias_inatividade": np.random.randint(0, 120, n),
        "total_gasto_acumulado_reais": np.random.uniform(0, 7500, n),
        "genero_cliente_m": np.random.binomial(1, 0.48, n),
        "regiao_cliente_nordeste": (regioes == 'nordeste').astype(int),
        "regiao_cliente_norte": (regioes == 'norte').astype(int),
        "regiao_cliente_sudeste": (regioes == 'sudeste').astype(int),
        "regiao_cliente_sul": (regioes == 'sul').astype(int)
    })
    
    # Target Treino
    prob_treino = 1 / (1 + np.exp(-(pdf_treino['renda_mensal_k']*0.1 + pdf_treino['total_gasto_acumulado_reais']*0.1 - 1400)))
    pdf_treino['comprou_eletronico'] = np.random.binomial(1, prob_treino)

    schema_dados = StructType([
        StructField("renda_mensal_k", DoubleType(), True),
        StructField("tempo_medio_clique_segundos", DoubleType(), True),
        StructField("media_interacoes_suporte", DoubleType(), True),
        StructField("media_cupons_ativos", DoubleType(), True),
        StructField("media_score_nps_cliente", DoubleType(), True),
        StructField("media_dias_inatividade", DoubleType(), True),
        StructField("total_gasto_acumulado_reais", DoubleType(), True),
        StructField("genero_cliente_m", IntegerType(), True),
        StructField("regiao_cliente_nordeste", IntegerType(), True),
        StructField("regiao_cliente_norte", IntegerType(), True),
        StructField("regiao_cliente_sudeste", IntegerType(), True),
        StructField("regiao_cliente_sul", IntegerType(), True),
        StructField("comprou_eletronico", IntegerType(), True)
    ])

    return spark.createDataFrame(pdf_treino, schema=schema_dados)

df_treino = gerar_treino(spark)

nome_tabela_treino = "workspace.default.base_treinamento_modelo"

df_treino.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_treino)

print(f"Tabela de treino gravada com sucesso em: {nome_tabela_treino}")